# tiny-voca Sinhala — 2. Train (GPU)

Aligns phonemes to the recordings, then trains the model. One Kaggle session runs
up to 12 hours; training saves before the limit and continues in the next session.

**First run**
1. Settings: **Accelerator: GPU T4 ×2** (one GPU is used) or **GPU P100**, **Internet: On**.
2. **Add Input** → *Your Work* → *Notebooks* → the output of **notebook 1** (data).
3. **Save Version → Save & Run All (Commit)**.

**Every next run (to continue training)**
1. **Add Input** → *Your Work* → *Notebooks* → **this notebook** (its latest version).
   Keep notebook 1's output attached too.
2. **Save Version → Save & Run All (Commit)** again. It finds the last checkpoint and resumes.
3. Do not change the training settings between runs, or resuming is refused.

Each run ends with audio samples (in the output's `samples/` folder and played below),
a pitch check of both voices, and an exported model in `model/`.

**Voice labels.** The first Path Nirvana release (3,300 clips) is Mrs. Oshadi (F1). Data built
with an early version of the data script labelled it M1; the relabelling cell below corrects
such data (it changes nothing in data built with the current script). To fine-tune an earlier
model instead of training from scratch, attach its output and set `INIT_FROM` to its run name.

In [ ]:
# Settings. Keep these the same in every run of the same training.
STEPS = 40000          # total training steps (the English model used 104,737 on a much faster GPU)
RUN_NAME = "sinhala"   # checkpoints of this training (resumes from here in later runs)
INIT_FROM = ""         # name of an earlier run to fine-tune from (e.g. "sinhala"), '' = train from scratch
GAN_START = 8000       # adversarial training starts here (sharper audio)
BATCH_FRAMES = 9000    # lower to 6000 if you get "CUDA out of memory"
SESSION_HOURS = 11.0   # stop and save before Kaggle's 12-hour limit
# Alignment quality gates, set from the Sinhala data: the aligner (trained without Sinhala) is less
# confident than on English, so its median clip scores ~0.44 / 0.375. These keep ~90% of clips.
MIN_TOKEN_PROBABILITY = 0.33
MAX_WEAK_PHONE_FRACTION = 0.5

In [ ]:
# Get tiny-voca-sinhala (tiny-voca with Sinhala support) at a fixed version and install what is missing.
import os, subprocess, sys, json, time, shutil, glob
from pathlib import Path
os.environ["TINY_VOCA_LANGUAGE"] = "si"          # switches tiny-voca to the Sinhala inventory
REPO = Path("/tmp/tiny-voca-sinhala")
COMMIT = "a1fc282ed60eb0294b5d352ea8c49e50412e417c"

def sh(command, quiet=False):
    """Run a shell command, show its output, stop the notebook if it fails."""
    print("$", command, flush=True)
    result = subprocess.run(command, shell=True, text=True, capture_output=quiet)
    if result.returncode:
        if quiet:
            print(result.stdout[-3000:], result.stderr[-3000:])
        raise SystemExit(f"command failed ({result.returncode}): {command}")

if not REPO.exists():
    sh(f"git clone -q https://github.com/Emindu/tiny-voca-sinhala {REPO}")
    sh(f"git -C {REPO} checkout -q {COMMIT}")
sh('pip install -q "phonemizer-fork==3.3.2" "espeakng-loader==0.2.4"', quiet=True)
sh(f"pip install -q --no-deps -e {REPO}", quiet=True)
sys.path.insert(0, str(REPO / "src"))
os.environ["PYTHONPATH"] = f"{REPO / 'src'}:{REPO / 'tests'}"
# "discover -s tests" imports the test as a top-level module: Kaggle has an unrelated package named "tests".
check = subprocess.run(f"cd {REPO} && python -m unittest discover -s tests -p test_frontend_sinhala.py",
                       shell=True, capture_output=True, text=True)
print(check.stderr[-1500:])
if check.returncode:
    raise SystemExit("the Sinhala front-end tests failed (see above)")

In [ ]:
# Find notebook 1's features and, if attached, this notebook's previous output
NOTEBOOK_START = time.time()
OUT = Path("/kaggle/working")

def find(marker):
    hits = sorted(Path("/kaggle/input").rglob(marker))
    return hits[0] if hits else None

features_marker = find("features/features.json")
if features_marker is None:
    raise SystemExit("Add notebook 1's output as input (Add Input -> Your Work -> Notebooks).")
FEATURES = features_marker.parent
CONFIG = FEATURES.parent / "data" / "config.json"
print("features:", FEATURES, "| voices:", json.loads(CONFIG.read_text())["speakers"])

previous = find("annotations/annotations.json")
if previous is not None and not (OUT / "annotations").exists():
    print("previous run found:", previous.parent.parent)
    for name in ("annotations", "annotations-fixed", "runs"):
        if (previous.parent.parent / name).exists():
            shutil.copytree(previous.parent.parent / name, OUT / name, dirs_exist_ok=True)
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on a GPU accelerator")
assert torch.cuda.is_available()
torch.zeros(1).cuda()   # fails early if this GPU is not supported by the installed PyTorch

In [ ]:
# Alignment: which frames belong to which phoneme. ~25 min the first time; changing the gates
# above only re-filters the saved alignments (a minute or two).
ANNOTATIONS = OUT / "annotations"
GATES = {"min_token_probability": MIN_TOKEN_PROBABILITY, "max_weak_phone_fraction": MAX_WEAK_PHONE_FRACTION}
saved = json.loads((ANNOTATIONS / "annotations.json").read_text()) if (ANNOTATIONS / "annotations.json").exists() else {}
if not saved.get("complete") or any(saved.get("limits", {}).get(k) != v for k, v in GATES.items()):
    from huggingface_hub import snapshot_download
    snapshot_download("facebook/wav2vec2-lv-60-espeak-cv-ft")
    sh(f"python -m tiny_voca.annotate --features {FEATURES} --output {ANNOTATIONS} --dataset-revision pathnirvana "
       f"--config {CONFIG} --device cuda --min-token-probability {MIN_TOKEN_PROBABILITY} "
       f"--max-weak-phone-fraction {MAX_WEAK_PHONE_FRACTION}" + (" --resume" if ANNOTATIONS.exists() else ""))
summary = json.loads((ANNOTATIONS / "summary.json").read_text())
for split, s in summary["splits"].items():
    total = s["accepted"] + s["rejected"]
    print(f"{split:10s} accepted {s['accepted']}/{total} ({s['accepted'] / max(total, 1):.0%}), "
          f"{s['accepted_hours']} h, by voice {s['accepted_by_voice']}, reasons {s['rejection_reasons']}")
train = summary["splits"]["train"]
if train["accepted"] / max(1, train["accepted"] + train["rejected"]) < 0.7 or not summary["splits"]["validation"]["accepted"]:
    q = summary["splits"]["train"]["qc_quantiles"]
    raise SystemExit("Too few clips passed the alignment gates. Medians: token probability "
                     f"{q['token_probability_mean']['50']}, weak phone fraction {q['weak_phone_fraction']['50']}. "
                     "Lower MIN_TOKEN_PROBABILITY / raise MAX_WEAK_PHONE_FRACTION in the settings cell and run again "
                     "(attach this run's output as input to skip re-aligning).")

In [ ]:
# Voice labels: the first Path Nirvana release (clip ids "pn-sin_01_...") is Mrs. Oshadi (F1).
# Earlier data builds labelled it M1 (Ven. Mettananda), which taught "M1" two voices: on some
# sentences it spoke at a female pitch. Training reads the voice from these manifests, so the
# fix needs no new features or alignment. Written once and reused, so resuming sees the same files.
from collections import Counter
LABELS = OUT / "annotations-fixed"
if not (LABELS / "annotations.json").exists():
    LABELS.mkdir(exist_ok=True)
    moved = Counter()
    for split in ("train", "validation", "test"):
        rows = [json.loads(line) for line in (ANNOTATIONS / f"{split}.jsonl").read_text(encoding="utf-8").split("\n") if line.strip()]
        for row in rows:
            if row["id"].startswith("pn-sin_01_") and row["speaker"] != "F1":
                row["speaker"] = "F1"
                moved[split] += 1
        (LABELS / f"{split}.jsonl").write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows), encoding="utf-8")
    shutil.copy(ANNOTATIONS / "annotations.json", LABELS / "annotations.json")
    print("relabelled M1 -> F1:", dict(moved))
for split in ("train", "validation", "test"):
    rows = [json.loads(line) for line in (LABELS / f"{split}.jsonl").read_text(encoding="utf-8").split("\n") if line.strip()]
    hours = Counter()
    for r in rows:
        hours[r["speaker"]] += r["frames"] * 256 / 24000 / 3600
    print(f"{split:10s} clips by voice {dict(Counter(r['speaker'] for r in rows))}, hours {({k: round(v, 2) for k, v in hours.items()})}")

In [ ]:
# Train until the session budget runs out; resumes automatically from the last checkpoint,
# or fine-tunes the earlier run (INIT_FROM) with the corrected voice labels.
RUN = OUT / "runs" / RUN_NAME
BASE = OUT / "runs" / INIT_FROM / "last.pt"
budget = int(SESSION_HOURS * 3600 - (time.time() - NOTEBOOK_START) - 20 * 60)   # leave 20 min for samples/export
command = [sys.executable, "-m", "tiny_voca.train", "--train", str(LABELS / "train.jsonl"),
           "--validation", str(LABELS / "validation.jsonl"), "--output", str(RUN), "--config", str(CONFIG),
           "--acoustic-root", str(FEATURES), "--device", "cuda", "--steps", str(STEPS), "--gan-start", str(GAN_START),
           "--gan-ratio", "0.7", "--teacher-final", "0", "--batch-frames", str(BATCH_FRAMES),
           "--max-seconds", str(budget), "--log-every", "50", "--save-every", "1000"]
if (RUN / "last.pt").exists():
    command += ["--resume", str(RUN / "last.pt")]
    print("resuming", RUN.name)
elif BASE.exists():
    command += ["--init-from", str(BASE), "--new-data"]
    print("fine-tuning", INIT_FROM, "->", RUN.name, "with the corrected voice labels")
elif INIT_FROM:
    raise SystemExit(f"runs/{INIT_FROM}/last.pt not found in the inputs. Attach the notebook version that trained it "
                     "(Add Input -> this notebook -> its latest successful version), or set INIT_FROM = '' to train from scratch.")
else:
    print("training", RUN.name, "from scratch")
print("training for up to", round(budget / 3600, 1), "hours")
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
tail = []
for line in process.stdout:
    tail = (tail + [line])[-40:]
    try:
        record = json.loads(line)
    except ValueError:
        if "Error" in line or "error" in line:
            print(line, end="")
        continue
    if "step" not in record:
        print(line, end="")
    elif record["step"] % 500 == 0 or "validation_predicted_mel" in record:
        keys = ("step", "loss", "mel", "validation_predicted_mel", "pitch_error_semitones", "steps_per_second")
        print({k: round(record[k], 4) if isinstance(record[k], float) else record[k] for k in keys if k in record}, flush=True)
if process.wait():
    print("".join(tail))
    raise SystemExit("training stopped with an error (shown above)")
last = json.loads(open(RUN / "metrics.jsonl").readlines()[-1])
remaining = max(0, STEPS - last["step"])
print(f"\nreached step {last['step']} of {STEPS}")
if remaining:
    hours = remaining / max(last["steps_per_second"], 1e-6) / 3600
    print(f"about {hours:.1f} GPU hours left: run this notebook again with this output attached as input")
else:
    print("training complete")

In [ ]:
# Listen: a few sentences in every voice, plus a pitch check of both voices on more sentences
import numpy as np, librosa, soundfile as sf
from IPython.display import Audio, display
from tiny_voca.infer import load_model, speak_text
SENTENCES = ["ආයුබෝවන්! ඔබට කොහොමද?",
             "අද කාලගුණය හොඳයි, ඒත් හවසට වැසි ඇති විය හැකියි.",
             "ශ්‍රී ලංකාව ඉන්දියන් සාගරයේ පිහිටි සුන්දර දිවයිනකි.",
             "මම හෙට උදේ කොළඹට යනවා."]
CHECK = SENTENCES + ["අද කාලගුණය හොඳයි.", "ඒත් හවසට වැසි ඇති විය හැකියි.",
                     "කරුණාකර මට උදව් කරන්න පුළුවන්ද?", "මගේ නම නිමල්, මම ගුරුවරයෙක්.",
                     "පාසල් නිවාඩුව ලබන සතියේ ආරම්භ වේ.", "බස් රථය උදේ හතට පිටත් වෙනවා, ඒ නිසා ඉක්මනට එන්න.",
                     "මේ පොත ඉතා රසවත්, ඔබත් කියවන්න.", "අපි හෙට ගමට යමු."]
model = load_model(RUN / "inference.pt", "cpu")
(OUT / "samples").mkdir(exist_ok=True)

def median_pitch(wave):
    f0, voiced, _ = librosa.pyin(wave.numpy().astype(np.float64), fmin=50, fmax=600, sr=model.config.sample_rate,
                                 frame_length=1024, hop_length=256)
    return float(np.median(f0[voiced])) if voiced.any() else float("nan")

pitch = {}
for voice in model.config.speakers:
    pitch[voice] = []
    for index, sentence in enumerate(CHECK):
        wave, _ = speak_text(model, sentence, voice)
        pitch[voice].append(median_pitch(wave))
        if index < len(SENTENCES):
            path = OUT / "samples" / f"{voice}-{index + 1}.wav"
            sf.write(path, wave.numpy(), model.config.sample_rate)
            print(voice, sentence, f"({pitch[voice][-1]:.0f} Hz)")
            display(Audio(str(path)))
print()
for voice, values in pitch.items():
    print(f"{voice}: median pitch per sentence {[f'{v:.0f}' for v in values]} Hz")
odd = [CHECK[i] for i, v in enumerate(pitch.get("M1", [])) if not v <= 165] + [CHECK[i] for i, v in enumerate(pitch.get("F1", [])) if not v >= 150]
print("PITCH CHECK OK: M1 stays male, F1 stays female" if not odd else f"PITCH CHECK: {len(odd)} sentence(s) out of range: {odd}")

In [ ]:
# Export: PyTorch checkpoint + ONNX for use anywhere
MODEL = OUT / "model"
MODEL.mkdir(exist_ok=True)
shutil.copy(RUN / "inference.pt", MODEL / "tiny-voca-si.pt")
sh(f"python -m tiny_voca.export --checkpoint {MODEL / 'tiny-voca-si.pt'} --output {MODEL / 'tiny-voca-si.onnx'}", quiet=True)
for path in sorted(MODEL.iterdir()):
    print(path.name, f"{path.stat().st_size / 1e6:.1f} MB")

## Use the model on your own computer

Download `model/tiny-voca-si.pt` from this notebook's output, then:

```bash
git clone https://github.com/Emindu/tiny-voca-sinhala && cd tiny-voca-sinhala
pip install -e ".[si]"
TINY_VOCA_LANGUAGE=si python -m tiny_voca.infer --checkpoint tiny-voca-si.pt --voice M1 \
  --text "ආයුබෝවන්! ඔබට කොහොමද?" --output hello.wav
```

It runs on a normal CPU. Voices: `M1` (male), `F1` (female). Or skip the code entirely and use
https://huggingface.co/eminduperera/tiny-voca-sinhala (ONNX + `speak.py`).